# Week 52, Day 4: N-queens and sudoku

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w052d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
n = 4
cells = [(r, c) for r in range(n) for c in range(n)]
for r, c in [(0, 1), (2, 3)]:
    print((r, c), "diagonals:", r - c, r + c)
print(len({r - c for r, c in cells}), len({r + c for r, c in cells}))

In [ ]:
def queens(n):
    cols, down, up = set(), set(), set()
    placed, out = [], []
    def place(r):
        if r == n:
            out.append(list(placed))
            return
        for c in range(n):
            if c in cols or r - c in down or r + c in up:
                continue
            cols.add(c); down.add(r - c); up.add(r + c); placed.append(c)
            place(r + 1)
            cols.discard(c); down.discard(r - c); up.discard(r + c); placed.pop()
    place(0)
    return out

solutions = queens(6)
print(len(solutions), solutions[0])
for c in solutions[0]:
    print(" ".join("Q" if i == c else "." for i in range(6)))

In [ ]:
def count_queens(n):
    full = (1 << n) - 1
    def solve(cols, down, up):
        if cols == full:
            return 1
        total = 0
        free = ~(cols | down | up) & full
        while free:
            bit = free & -free                  # the lowest safe column
            free -= bit
            total += solve(cols | bit, ((down | bit) << 1) & full, (up | bit) >> 1)
        return total
    return solve(0, 0, 0)

print([count_queens(n) for n in range(1, 11)])

In [ ]:
def candidates(grid, r, c):
    used = set(grid[r]) | {grid[i][c] for i in range(9)}
    br, bc = r // 3 * 3, c // 3 * 3
    used |= {grid[i][j] for i in range(br, br + 3) for j in range(bc, bc + 3)}
    return [d for d in range(1, 10) if d not in used]

puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]
print(candidates(puzzle, 0, 2), candidates(puzzle, 4, 4))

In [ ]:
def solve(grid):
    rows = [set(row) - {0} for row in grid]
    cols = [{grid[r][c] for r in range(9)} - {0} for c in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            if grid[r][c]:
                boxes[r // 3 * 3 + c // 3].add(grid[r][c])
    guesses = [0]

    def fill():
        best = None
        for r in range(9):
            for c in range(9):
                if grid[r][c] == 0:
                    opts = set(range(1, 10)) - rows[r] - cols[c] - boxes[r // 3 * 3 + c // 3]
                    if best is None or len(opts) < len(best[2]):
                        best = (r, c, opts)
        if best is None:
            return True                                   # no empty cell: solved
        r, c, opts = best
        for d in sorted(opts):
            guesses[0] += 1
            b = r // 3 * 3 + c // 3
            grid[r][c] = d; rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if fill():
                return True
            grid[r][c] = 0; rows[r].discard(d); cols[c].discard(d); boxes[b].discard(d)
        return False

    return fill(), guesses[0]

puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0], [6, 0, 0, 1, 9, 5, 0, 0, 0], [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3], [4, 0, 0, 8, 0, 3, 0, 0, 1], [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0], [0, 0, 0, 4, 1, 9, 0, 0, 5], [0, 0, 0, 0, 8, 0, 0, 7, 9],
]
print(solve(puzzle), puzzle[0])

> **Key idea.** N-queens: one queen per row, and a square is safe when its column, its r − c diagonal and its r + c diagonal are free, three O(1) set checks. Bitmasks make counting faster still: `free & -free` picks the lowest safe column, and shifting the diagonal masks moves them to the next row. Sudoku: each cell's candidates are the digits not in its row, column or box (box `r // 3 * 3 + c // 3`). Always fill the most constrained cell next (fewest candidates): forced moves cost nothing and dead ends show up at once. Propagate consequences (naked and hidden singles) to guess even less.

### Exercise: Queens with sets

Write `queens(n)`: every solution, each a list of the queens' columns row by row, in the order a search trying columns left to right finds them, with three sets for O(1) checks. Then `draw(solution)`: the board as a list of strings, `Q` for a queen and `.` otherwise.

In [ ]:
def queens(n):
    ___

def draw(solution):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queens():\n    \"queens\"\n    assert queens(4) == [[1, 3, 0, 2], [2, 0, 3, 1]] and queens(1) == [[0]] and queens(3) == []\n    assert len(queens(6)) == 4 and len(queens(8)) == 92\n    for sol in queens(7):\n        n = len(sol)\n        assert len(set(sol)) == n and len({r - c for r, c in enumerate(sol)}) == n and len({r + c for r, c in enumerate(sol)}) == n\n    assert draw([1, 3, 0, 2]) == [\".Q..\", \"...Q\", \"Q...\", \"..Q.\"]")

### Exercise: Queens with bits

Write `count_queens(n)` with bitmasks as in "Queens with bits": no sets or lists in the search. Then `first_solution(n)`: the first solution as a list of columns (lowest safe column first in each row), or `None`, also with bitmasks (`bit.bit_length() - 1` turns a single bit back into a column number). The checks count up to n = 11 within a few seconds.

In [ ]:
def count_queens(n):
    ___

def first_solution(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\ndef test_bits():\n    \"bits\"\n    assert [count_queens(n) for n in range(1, 9)] == [1, 0, 0, 2, 10, 4, 40, 92]\n    assert first_solution(4) == [1, 3, 0, 2] and first_solution(3) is None and first_solution(8) == [0, 4, 7, 5, 2, 6, 1, 3]\n    t0 = time.perf_counter()\n    assert count_queens(11) == 2680 and time.perf_counter() - t0 < 8")

### Exercise: Checking a sudoku

Grids are 9 lists of 9 numbers, 0 for empty. Write `is_valid(grid)`: whether no row, column or box contains a non-zero digit twice. Then `candidates(grid, r, c)`: the sorted list of digits that could go in empty cell (r, c) (`[]` if it's filled), and `is_solved(grid)`: valid with no empty cells.

In [ ]:
def is_valid(grid):
    ___

def candidates(grid, r, c):
    ___

def is_solved(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PUZZLE = [\n    [5, 3, 0, 0, 7, 0, 0, 0, 0], [6, 0, 0, 1, 9, 5, 0, 0, 0], [0, 9, 8, 0, 0, 0, 0, 6, 0],\n    [8, 0, 0, 0, 6, 0, 0, 0, 3], [4, 0, 0, 8, 0, 3, 0, 0, 1], [7, 0, 0, 0, 2, 0, 0, 0, 6],\n    [0, 6, 0, 0, 0, 0, 2, 8, 0], [0, 0, 0, 4, 1, 9, 0, 0, 5], [0, 0, 0, 0, 8, 0, 0, 7, 9],\n]\nSOLVED = [\n    [5, 3, 4, 6, 7, 8, 9, 1, 2], [6, 7, 2, 1, 9, 5, 3, 4, 8], [1, 9, 8, 3, 4, 2, 5, 6, 7],\n    [8, 5, 9, 7, 6, 1, 4, 2, 3], [4, 2, 6, 8, 5, 3, 7, 9, 1], [7, 1, 3, 9, 2, 4, 8, 5, 6],\n    [9, 6, 1, 5, 3, 7, 2, 8, 4], [2, 8, 7, 4, 1, 9, 6, 3, 5], [3, 4, 5, 2, 8, 6, 1, 7, 9],\n]\ndef test_valid():\n    \"checking\"\n    assert is_valid(PUZZLE) and is_valid(SOLVED) and is_solved(SOLVED) and not is_solved(PUZZLE)\n    bad = [row[:] for row in PUZZLE]\n    bad[0][2] = 5\n    assert not is_valid(bad)\n    bad = [row[:] for row in PUZZLE]\n    bad[1][1] = 8\n    assert not is_valid(bad), \"box clash\"\n    assert candidates(PUZZLE, 0, 2) == [1, 2, 4] and candidates(PUZZLE, 4, 4) == [5] and candidates(PUZZLE, 0, 0) == []")

### Exercise: A sudoku solver

Write `solve(grid)`: fill the grid **in place** and return `True`, or return `False` (leaving the grid as it was) if it has no solution, by backtracking on the **most constrained** empty cell each time, keeping used-digit sets for rows, columns and boxes. Also count placements in `solve.placements`, an attribute set on the function (start it at 0 at the beginning of each call). The checks include a hard puzzle that a fixed-order search finds slow.

In [ ]:
def solve(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\nPUZZLE = [\n    [5, 3, 0, 0, 7, 0, 0, 0, 0], [6, 0, 0, 1, 9, 5, 0, 0, 0], [0, 9, 8, 0, 0, 0, 0, 6, 0],\n    [8, 0, 0, 0, 6, 0, 0, 0, 3], [4, 0, 0, 8, 0, 3, 0, 0, 1], [7, 0, 0, 0, 2, 0, 0, 0, 6],\n    [0, 6, 0, 0, 0, 0, 2, 8, 0], [0, 0, 0, 4, 1, 9, 0, 0, 5], [0, 0, 0, 0, 8, 0, 0, 7, 9],\n]\nHARD = [[int(ch) for ch in row] for row in [\n    \"800000000\", \"003600000\", \"070090200\", \"050007000\", \"000045700\",\n    \"000100030\", \"001000068\", \"008500010\", \"090000400\"]]\ndef ok(g):\n    groups = [g[i] for i in range(9)] + [[g[r][i] for r in range(9)] for i in range(9)]\n    groups += [[g[r][c] for r in range(b // 3 * 3, b // 3 * 3 + 3) for c in range(b % 3 * 3, b % 3 * 3 + 3)] for b in range(9)]\n    return all(sorted(x) == list(range(1, 10)) for x in groups)\ndef test_solver():\n    \"solving\"\n    g = [row[:] for row in PUZZLE]\n    assert solve(g) and ok(g) and all(g[r][c] == PUZZLE[r][c] for r in range(9) for c in range(9) if PUZZLE[r][c])\n    assert solve.placements == 51\n    t0 = time.perf_counter()\n    h = [row[:] for row in HARD]\n    assert solve(h) and ok(h) and time.perf_counter() - t0 < 10\n    bad = [[0] * 9 for _ in range(9)]\n    bad[0][:8] = [1, 2, 3, 4, 5, 6, 7, 8]\n    bad[1][8] = 9                        # cell (0, 8) needs 9, but its column already has one\n    copy = [row[:] for row in bad]\n    assert solve(bad) is False and bad == copy")

### Exercise: Forced moves

Write `naked_singles(grid)`: repeatedly fill every empty cell that has exactly **one** candidate, until none is left, filling the grid in place and returning how many cells it filled. Then `hidden_singles(grid)`: one pass that, for each row, column and box, places any digit that fits in exactly one of its empty cells; return how many it placed. Many easy puzzles fall to these alone.

In [ ]:
def naked_singles(grid):
    ___

def hidden_singles(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PUZZLE = [\n    [5, 3, 0, 0, 7, 0, 0, 0, 0], [6, 0, 0, 1, 9, 5, 0, 0, 0], [0, 9, 8, 0, 0, 0, 0, 6, 0],\n    [8, 0, 0, 0, 6, 0, 0, 0, 3], [4, 0, 0, 8, 0, 3, 0, 0, 1], [7, 0, 0, 0, 2, 0, 0, 0, 6],\n    [0, 6, 0, 0, 0, 0, 2, 8, 0], [0, 0, 0, 4, 1, 9, 0, 0, 5], [0, 0, 0, 0, 8, 0, 0, 7, 9],\n]\nSOLVED = [\n    [5, 3, 4, 6, 7, 8, 9, 1, 2], [6, 7, 2, 1, 9, 5, 3, 4, 8], [1, 9, 8, 3, 4, 2, 5, 6, 7],\n    [8, 5, 9, 7, 6, 1, 4, 2, 3], [4, 2, 6, 8, 5, 3, 7, 9, 1], [7, 1, 3, 9, 2, 4, 8, 5, 6],\n    [9, 6, 1, 5, 3, 7, 2, 8, 4], [2, 8, 7, 4, 1, 9, 6, 3, 5], [3, 4, 5, 2, 8, 6, 1, 7, 9],\n]\ndef test_singles():\n    \"propagation\"\n    g = [row[:] for row in PUZZLE]\n    filled = naked_singles(g)\n    assert filled > 0 and all(g[r][c] in (0, SOLVED[r][c]) for r in range(9) for c in range(9))\n    g = [row[:] for row in PUZZLE]\n    placed = hidden_singles(g)\n    assert placed > 0 and all(g[r][c] in (0, SOLVED[r][c]) for r in range(9) for c in range(9))\n    g = [row[:] for row in PUZZLE]\n    for _ in range(10):\n        naked_singles(g)\n        hidden_singles(g)\n    assert g == SOLVED, \"this puzzle needs no guessing\"")

### Bug hunt: Half the diagonals

This queen search checks columns and the r + c diagonals but forgets the r − c ones, so queens attack each other along "\\" diagonals and it finds far too many "solutions": this prints `7` for n = 4. Track r − c too so it prints `2`.

In [ ]:
def count(n):
    cols, up = set(), set()
    def place(r):
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or r + c in up:
                continue
            cols.add(c)
            up.add(r + c)
            total += place(r + 1)
            cols.discard(c)
            up.discard(r + c)
        return total
    return place(0)

print(count(4))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both diagonals\"\n    expect_output(\"2\")")

### Bug hunt: The wrong box

This numbers boxes with `r // 3 + c // 3`, so different boxes get the same number (box (0, 1) and box (1, 0) are both 1) and digits clash wrongly: a valid grid is reported invalid and this prints `False`. Use `r // 3 * 3 + c // 3` so it prints `True`.

In [ ]:
SOLVED = [
    [5, 3, 4, 6, 7, 8, 9, 1, 2], [6, 7, 2, 1, 9, 5, 3, 4, 8], [1, 9, 8, 3, 4, 2, 5, 6, 7],
    [8, 5, 9, 7, 6, 1, 4, 2, 3], [4, 2, 6, 8, 5, 3, 7, 9, 1], [7, 1, 3, 9, 2, 4, 8, 5, 6],
    [9, 6, 1, 5, 3, 7, 2, 8, 4], [2, 8, 7, 4, 1, 9, 6, 3, 5], [3, 4, 5, 2, 8, 6, 1, 7, 9],
]
boxes = {}
ok = True
for r in range(9):
    for c in range(9):
        b = r // 3 + c // 3
        if SOLVED[r][c] in boxes.setdefault(b, set()):
            ok = False
        boxes[b].add(SOLVED[r][c])
print(ok)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"nine boxes\"\n    expect_output(\"True\")")

### Bug hunt: Diagonals that don't move

In the bitmask counter, the "\\" diagonal mask moves one column left each row and the "/" mask one column right. This version shifts **both** masks left, so the "/" attacks drift the wrong way and real attacks are missed: this prints `7` for n = 4. Shift `up` to the right so it prints `2`.

In [ ]:
def count_queens(n):
    full = (1 << n) - 1
    def solve(cols, down, up):
        if cols == full:
            return 1
        total = 0
        free = ~(cols | down | up) & full
        while free:
            bit = free & -free
            free -= bit
            total += solve(cols | bit, ((down | bit) << 1) & full, ((up | bit) << 1) & full)
        return total
    return solve(0, 0, 0)

print(count_queens(4))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"opposite directions\"\n    expect_output(\"2\")")

### Puzzle: Placing queens

Arrange the lines into the set-based queen search. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of N-queens solutions for n = 5
q = ___   # the r - c and r + c of square (2, 5), as a tuple
r = ___   # the box number of sudoku cell (4, 7)
s = ___   # 12 & -12 (the lowest set bit of 12)
t = ___   # the number of "\" diagonals on an 8 x 8 board

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 10 and q == (-3, 7) and r == 5, \"p, q and r\"\n    assert s == 4 and t == 15, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: exact cover and Algorithm X

Many puzzles are **exact cover** problems: given a collection of subsets of some "constraints", choose subsets so that every constraint is covered **exactly once**. Sudoku is one: choosing "digit d in cell (r, c)" covers four constraints: cell (r, c) is filled, row r has d, column c has d, box b has d. All 324 constraints must each be covered once.

Donald Knuth's **Algorithm X** (2000) solves exact cover by backtracking: pick the constraint with the **fewest** subsets that cover it (most constrained first, again); try each such subset; select it by removing every constraint it covers and every subset that clashes with it; recurse; then restore. Ali Assaf's well-known short Python version stores it as two dicts: `X`, from each constraint to the **set** of subset names covering it, and `Y`, from each subset name to its list of constraints. Write:

- `solve_cover(X, Y)`: a list of subset names forming an exact cover, or `None`, using select and deselect helpers that change `X` and restore it exactly;
- `sudoku_cover(grid)`: build X and Y for sudoku (subset names `(r, c, d)`), select the givens first, solve, and return the completed grid (a new list of lists), or `None`.

In [ ]:
def solve_cover(X, Y):
    ___

def sudoku_cover(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\nHARD = [[int(ch) for ch in row] for row in [\n    \"800000000\", \"003600000\", \"070090200\", \"050007000\", \"000045700\",\n    \"000100030\", \"001000068\", \"008500010\", \"090000400\"]]\ndef ok(g):\n    groups = [g[i] for i in range(9)] + [[g[r][i] for r in range(9)] for i in range(9)]\n    groups += [[g[r][c] for r in range(b // 3 * 3, b // 3 * 3 + 3) for c in range(b % 3 * 3, b % 3 * 3 + 3)] for b in range(9)]\n    return all(sorted(x) == list(range(1, 10)) for x in groups)\ndef test_cover():\n    \"exact cover\"\n    Y = {\"A\": [1, 4, 7], \"B\": [1, 4], \"C\": [4, 5, 7], \"D\": [3, 5, 6], \"E\": [2, 3, 6, 7], \"F\": [2, 7]}\n    X = {c: {name for name, cs in Y.items() if c in cs} for c in range(1, 8)}\n    assert sorted(solve_cover(X, Y)) == [\"B\", \"D\", \"F\"]\n    Y2 = {\"A\": [1, 2], \"B\": [2, 3]}\n    X2 = {c: {n for n, cs in Y2.items() if c in cs} for c in (1, 2, 3)}\n    assert solve_cover(X2, Y2) is None\ndef test_sudoku():\n    \"sudoku\"\n    t0 = time.perf_counter()\n    g = sudoku_cover(HARD)\n    assert ok(g) and all(g[r][c] == HARD[r][c] for r in range(9) for c in range(9) if HARD[r][c])\n    assert time.perf_counter() - t0 < 10\n    bad = [row[:] for row in HARD]\n    bad[0][1] = 8\n    assert sudoku_cover(bad) is None")

## Warm-up

Every square on one "\\" diagonal has the same r − c, and every square on one "/" diagonal the same r + c. So a board has 2n − 1 diagonals each way, and "is this diagonal taken?" is a set lookup.

## Queens with sets

Place one queen per row (two in a row would attack), trying each column. A square is safe if its **column**, its **r − c** diagonal and its **r + c** diagonal are all free. Keep three sets of what's taken: each check is O(1), and placing or removing a queen is three set operations, which is the choose and unchoose of Wednesday's recipe.

Week 16 checked each new queen against every queen already placed, O(n) per check; the sets make it O(1). The number of solutions grows quickly (92 for the standard 8 × 8 board, 14,200 for 12 × 12), and the search visits far more partial boards than solutions.

## Queens with bits

To **count** solutions fast, replace the three sets with three integers used as **bitmasks** (Week 51 Day 5): bit c of `cols` is 1 if column c is taken. Better still, shift the diagonal masks as you go down a row, so that bit c of each mask always means "attacked in **this** row":

- `cols | down | up` has a 1 for every attacked column in this row; `free = ~(cols | down | up) & full` has a 1 for every safe one (`full` is n ones, `(1 << n) - 1`).
- `free & -free` isolates the **lowest** set bit: one safe column.
- Going down a row, a "\\" diagonal moves one column right and a "/" one moves left: pass `(down | bit) << 1` and `(up | bit) >> 1`.

Each step is a handful of integer operations, with no sets, lists or loops over columns. This version is often credited to Martin Richards, who published it in 1997.

## Sudoku as a search

A sudoku grid has 81 cells in nine rows, nine columns and nine 3 × 3 **boxes**; every row, column and box must contain each digit 1 to 9 once. The box of cell (r, c) is number `(r // 3) * 3 + c // 3`.

A cell's **candidates** are the digits not already in its row, column or box. Keep a set of used digits for each row, column and box, and the candidates are a set difference. Plain backtracking: find an empty cell, try each candidate, recurse, undo. It works, but can make millions of guesses on hard puzzles if it fills cells in a fixed order.

## Most constrained first

The most effective single improvement to a backtracking search is choosing **which** decision to make next: always fill the empty cell with the **fewest candidates** (the "minimum remaining values" heuristic). A cell with one candidate is a forced move, no guessing at all; a cell with none means the current path is already doomed, and the search backs up at once instead of filling other cells first. On real puzzles, this cuts the search from millions of steps to hundreds.

51 empty cells, 51 placements: every choice on this puzzle was forced once the most constrained cell came first.

## Propagation

People solve sudoku mostly by **propagation**: placing a digit removes it from the candidates of its row, column and box, which may leave another cell with a single candidate (a "naked single"), which is placed in turn, and so on. Another rule: if a digit can go in only **one** cell of a row, column or box (a "hidden single"), it must go there. Running propagation before (and after) each guess means the search guesses far less. Peter Norvig's well-known essay "Solving Every Sudoku Puzzle" combines exactly these two rules with most-constrained-first search; this lesson's approach follows its ideas.

Constraint propagation is a general technique far beyond puzzles: timetabling, configuration tools, and the solvers used in planning and chip design all repeatedly apply "this choice forces that" before searching.

## Practice

## Quiz

## Daily challenge

## Before you go

Solve a newspaper sudoku by hand, and notice which of today's rules you use: naked singles, hidden singles, and, when stuck, a guess on the cell with fewest options. Tomorrow: pruning, the art of cutting the search tree down, with bounds that rule out whole branches.